# Fixed PINN for 2D INS — v2
### Changes from v1:
1. **Gravity compensation** — subtract gravity from IMU accel using full 3D orientation
2. **Output normalization** — network predicts in normalized space
3. **Per-residual weighting** — physics residuals balanced by characteristic scale
4. **Lambda annealing** — physics weight starts small and ramps up
5. **Evaluate function fixed** — was referencing undefined `px_mean` etc.

In [28]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation

# ── Load raw data ──
base = "/kaggle/input/datasets/bhavaniprasadm3/euroc-mav/MH_01_easy"

imu_df = pd.read_csv(f"{base}/mav0/imu0/data.csv")
gt_df  = pd.read_csv(f"{base}/mav0/state_groundtruth_estimate0/data.csv")

In [29]:
# ── Align timestamps ──
gt_times = gt_df["#timestamp"].values
imu_times = imu_df["#timestamp [ns]"].values

indices = np.searchsorted(imu_times, gt_times, side="left")
indices = np.clip(indices, 0, len(imu_times) - 1)
imu_aligned = imu_df.iloc[indices].reset_index(drop=True)

## FIX 1 — Gravity compensation

The IMU accelerometer measures **specific force** = true\_acceleration + gravity\_in\_sensor\_frame.  
Before using `ax, ay` in our 2D physics model, we must **rotate the gravity vector `[0, 0, g]`
into the sensor frame using the full 3D orientation** and subtract it.

Without this, the physics residuals for `v̇x` and `v̇y` try to match ~9.8 m/s² of spurious
acceleration and can **never converge** — that's why physics loss was stuck at ~80.

In [30]:
# ── Raw 3D IMU accelerations ──
ax_raw = imu_aligned["a_RS_S_x [m s^-2]"].values.astype(np.float64)
ay_raw = imu_aligned["a_RS_S_y [m s^-2]"].values.astype(np.float64)
az_raw = imu_aligned["a_RS_S_z [m s^-2]"].values.astype(np.float64)
wz = imu_aligned["w_RS_S_z [rad s^-1]"].values.astype(np.float32)

# ── Full 3D orientation from ground-truth quaternions ──
quats = gt_df[[" q_RS_x []", " q_RS_y []", " q_RS_z []", " q_RS_w []"]].values
rotations = Rotation.from_quat(quats)           # world-to-sensor rotation

# ── Subtract gravity ──
# Gravity in world frame: [0, 0, 9.81]
# In sensor frame: R_sensor_from_world @ [0, 0, 9.81]
g_world = np.array([0.0, 0.0, 9.81])
g_sensor = rotations.inv().apply(g_world)        # (N, 3) gravity in sensor frame
#   inv() because EuRoC quaternions are R_world_from_sensor (q_RS),
#   so inv gives R_sensor_from_world
#   Actually EuRoC q_RS maps sensor→world, so R_sensor_from_world = inv(q_RS)
#   Let's verify: for a level hover, az_raw ≈ +9.81, g_sensor_z ≈ +9.81 → subtraction ≈ 0 ✓

# Corrected accelerations (true body-frame acceleration, gravity-free)
ax_corr = (ax_raw - g_sensor[:, 0]).astype(np.float32)
ay_corr = (ay_raw - g_sensor[:, 1]).astype(np.float32)

print(f"Before gravity comp — ax mean: {ax_raw.mean():.2f}, ay mean: {ay_raw.mean():.2f}, az mean: {az_raw.mean():.2f}")
print(f"After  gravity comp — ax mean: {ax_corr.mean():.2f}, ay mean: {ay_corr.mean():.2f}")
print(f"(Should be near 0 for a dataset with roughly symmetric motion)")

Before gravity comp — ax mean: 9.11, ay mean: -0.07, az mean: -3.36
After  gravity comp — ax mean: -0.02, ay mean: 0.13
(Should be near 0 for a dataset with roughly symmetric motion)


In [31]:
# ── Ground truth ──
px_gt = gt_df[" p_RS_R_x [m]"].values.astype(np.float32)
py_gt = gt_df[" p_RS_R_y [m]"].values.astype(np.float32)
vx_gt = gt_df[" v_RS_R_x [m s^-1]"].values.astype(np.float32)
vy_gt = gt_df[" v_RS_R_y [m s^-1]"].values.astype(np.float32)
yaw_gt = Rotation.from_quat(quats).as_euler("ZYX")[:, 0].astype(np.float32)

# ── Time in seconds from 0 ──
t = ((gt_times - gt_times[0]) / 1e9).astype(np.float32)

# Use gravity-corrected accel from now on
ax = ax_corr
ay = ay_corr

## FIX 2 — Normalize BOTH inputs AND outputs

v1 only normalized inputs. The network with `tanh` activations had to learn wildly different
output scales (position ~meters vs yaw ~radians) through a single linear layer.  
Now we normalize everything, train in normalized space, and denormalize for evaluation.

In [32]:
# ── Normalize inputs ──
t_mean, t_std   = t.mean(), t.std()
ax_mean, ax_std = ax.mean(), ax.std()
ay_mean, ay_std = ay.mean(), ay.std()
wz_mean, wz_std = wz.mean(), wz.std()

t_norm  = (t  - t_mean)  / t_std
ax_norm = (ax - ax_mean) / ax_std
ay_norm = (ay - ay_mean) / ay_std
wz_norm = (wz - wz_mean) / wz_std

# ── FIX: Normalize outputs too ──
px_mean, px_std   = px_gt.mean(), px_gt.std()
py_mean, py_std   = py_gt.mean(), py_gt.std()
vx_mean, vx_std   = vx_gt.mean(), vx_gt.std()
vy_mean, vy_std   = vy_gt.mean(), vy_gt.std()
yaw_mean, yaw_std = yaw_gt.mean(), yaw_gt.std()

px_norm  = (px_gt  - px_mean)  / px_std
py_norm  = (py_gt  - py_mean)  / py_std
vx_norm  = (vx_gt  - vx_mean)  / vx_std
vy_norm  = (vy_gt  - vy_mean)  / vy_std
yaw_norm = (yaw_gt - yaw_mean) / yaw_std

print(f"Output stats:")
print(f"  px:  mean={px_mean:.3f}  std={px_std:.3f}")
print(f"  py:  mean={py_mean:.3f}  std={py_std:.3f}")
print(f"  vx:  mean={vx_mean:.3f}  std={vx_std:.3f}")
print(f"  vy:  mean={vy_mean:.3f}  std={vy_std:.3f}")
print(f"  yaw: mean={yaw_mean:.3f}  std={yaw_std:.3f}")

Output stats:
  px:  mean=2.170  std=2.381
  py:  mean=2.711  std=3.533
  vx:  mean=-0.001  std=0.300
  vy:  mean=-0.001  std=0.389
  yaw: mean=1.782  std=1.775


In [33]:
# ── Build tensors (now targets are NORMALIZED) ──
X = torch.tensor(np.stack([t_norm, ax_norm, ay_norm, wz_norm], axis=1))
Y = torch.tensor(np.stack([px_norm, py_norm, vx_norm, vy_norm, yaw_norm], axis=1))
T = torch.tensor(t_norm).unsqueeze(1).requires_grad_(True)

# ── Train / Val / Test split (70/15/15 by time) ──
N = len(X)
n_train, n_val = int(0.7 * N), int(0.15 * N)

X_train, Y_train, T_train = X[:n_train], Y[:n_train], T[:n_train]
X_val,   Y_val,   T_val   = X[n_train:n_train+n_val], Y[n_train:n_train+n_val], T[n_train:n_train+n_val]
X_test,  Y_test,  T_test  = X[n_train+n_val:], Y[n_train+n_val:], T[n_train+n_val:]

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Train: 25467 | Val: 5457 | Test: 5458


## FIX 3 — Physics loss with residual balancing

Two problems in v1:
1. All 5 residuals were summed raw — but `r3, r4` (acceleration-scale) were ~100× larger than `r1, r2` (velocity-scale), so the optimizer only saw gradients from the acceleration equations.
2. The physics operated on raw outputs but the network now predicts in normalized space.

We now convert network outputs back to real-world units, compute residuals, then **divide each
residual by its characteristic scale** so they contribute equally.

In [34]:
def physics_loss(pred_norm, inputs_t, imu_data_norm, norm_p):
    """
    pred_norm:      (N,5) NORMALIZED network output
    inputs_t:       (N,1) NORMALIZED time with grad
    imu_data_norm:  (N,3) NORMALIZED [ax, ay, wz]
    norm_p:         dict of all normalization params (on same device)
    """
    # ── Denormalize predictions to real-world ──
    px  = pred_norm[:, 0:1] * norm_p["px_std"]  + norm_p["px_mean"]
    py  = pred_norm[:, 1:2] * norm_p["py_std"]  + norm_p["py_mean"]
    vx  = pred_norm[:, 2:3] * norm_p["vx_std"]  + norm_p["vx_mean"]
    vy  = pred_norm[:, 3:4] * norm_p["vy_std"]  + norm_p["vy_mean"]
    psi = pred_norm[:, 4:5] * norm_p["yaw_std"] + norm_p["yaw_mean"]

    # ── Time derivatives (chain rule: d/dt_real = d/dt_norm / t_std) ──
    ones = torch.ones_like(px)
    dpx_dt  = torch.autograd.grad(px,  inputs_t, ones, create_graph=True)[0] / norm_p["t_std"]
    dpy_dt  = torch.autograd.grad(py,  inputs_t, ones, create_graph=True)[0] / norm_p["t_std"]
    dvx_dt  = torch.autograd.grad(vx,  inputs_t, ones, create_graph=True)[0] / norm_p["t_std"]
    dvy_dt  = torch.autograd.grad(vy,  inputs_t, ones, create_graph=True)[0] / norm_p["t_std"]
    dpsi_dt = torch.autograd.grad(psi, inputs_t, ones, create_graph=True)[0] / norm_p["t_std"]

    # ── Denormalize IMU to real-world (already gravity-compensated!) ──
    ax_real = imu_data_norm[:, 0:1] * norm_p["ax_std"] + norm_p["ax_mean"]
    ay_real = imu_data_norm[:, 1:2] * norm_p["ay_std"] + norm_p["ay_mean"]
    wz_real = imu_data_norm[:, 2:3] * norm_p["wz_std"] + norm_p["wz_mean"]

    # ── Residuals ──
    r1 = dpx_dt - vx                                                    # ṗx = vx
    r2 = dpy_dt - vy                                                    # ṗy = vy
    r3 = dvx_dt - (ax_real * torch.cos(psi) - ay_real * torch.sin(psi)) # v̇x
    r4 = dvy_dt - (ax_real * torch.sin(psi) + ay_real * torch.cos(psi)) # v̇y
    r5 = dpsi_dt - wz_real                                              # ψ̇ = wz

    # ── FIX: Normalize each residual by its characteristic scale ──
    # so they all contribute roughly equally to the gradient
    vel_scale  = max(float(norm_p["vx_std"]), float(norm_p["vy_std"]), 0.1)
    acc_scale  = max(float(norm_p["ax_std"]), float(norm_p["ay_std"]), 0.1)
    yaw_scale  = max(float(norm_p["wz_std"]), 0.1)

    loss = (
        (r1 / vel_scale).pow(2).mean() +
        (r2 / vel_scale).pow(2).mean() +
        (r3 / acc_scale).pow(2).mean() +
        (r4 / acc_scale).pow(2).mean() +
        (r5 / yaw_scale).pow(2).mean()
    ) / 5.0

    return loss

In [35]:
class VanillaPINN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        layers = []
        dims = [4, 128, 128, 128, 128, 128, 5]
        for i in range(len(dims) - 1):
            layers.append(torch.nn.Linear(dims[i], dims[i+1]))
            if i < len(dims) - 2:
                layers.append(torch.nn.Tanh())
        self.net = torch.nn.Sequential(*layers)

    def forward(self, x, t):
        inp = torch.cat([t, x[:, 1:4]], dim=1)
        return self.net(inp)

## FIX 4 — Lambda annealing in training loop

Instead of a fixed `λ=0.1` from epoch 0, we **ramp λ from 0 → target** over a warmup period.  
This lets the network first learn a reasonable data fit, then gradually enforce physics.  
With incorrect/noisy physics, a large λ from the start poisons early training.

In [36]:
def train_model(model, X_tr, Y_tr, T_tr, epochs=5000, lr=5e-4,
                lam=0.1, arch="vanilla", norm_p=None,
                lam_warmup=1000):
    """
    lam_warmup: epochs over which lambda ramps from 0 → lam
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)

    X_tr = X_tr.to(device)
    Y_tr = Y_tr.to(device)
    T_tr = T_tr.to(device).requires_grad_(True)

    history = {"epoch": [], "data_loss": [], "phys_loss": [], "total_loss": []}

    for ep in range(epochs):
        optimizer.zero_grad()

        pred = model(X_tr, T_tr)
        imu_for_phys = X_tr[:, 1:4]

        L_data = torch.nn.functional.mse_loss(pred, Y_tr)
        L_phys = physics_loss(pred, T_tr, imu_for_phys, norm_p)

        # ── FIX: Ramp lambda from 0 → lam over warmup epochs ──
        lam_cur = lam * min(1.0, ep / max(lam_warmup, 1))

        L_total = L_data + lam_cur * L_phys
        L_total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        if ep % 500 == 0:
            print(f"[{arch}] Ep {ep:4d} | Data: {L_data:.6f} | "
                  f"Phys: {L_phys:.6f} | λ_cur: {lam_cur:.4f} | Total: {L_total:.6f}")

        history["epoch"].append(ep)
        history["data_loss"].append(L_data.item())
        history["phys_loss"].append(L_phys.item())
        history["total_loss"].append(L_total.item())

    return model, history

In [37]:
# ── Pack ALL normalization params into one dict on device ──
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def to_t(x):
    return torch.tensor(x, dtype=torch.float32, device=device)

norm_p = {
    "t_std":    to_t(t_std),
    "ax_mean":  to_t(ax_mean),  "ax_std":  to_t(ax_std),
    "ay_mean":  to_t(ay_mean),  "ay_std":  to_t(ay_std),
    "wz_mean":  to_t(wz_mean),  "wz_std":  to_t(wz_std),
    "px_mean":  to_t(px_mean),  "px_std":  to_t(px_std),
    "py_mean":  to_t(py_mean),  "py_std":  to_t(py_std),
    "vx_mean":  to_t(vx_mean),  "vx_std":  to_t(vx_std),
    "vy_mean":  to_t(vy_mean),  "vy_std":  to_t(vy_std),
    "yaw_mean": to_t(yaw_mean), "yaw_std": to_t(yaw_std),
}

In [40]:



model1 = VanillaPINN()
model1, hist1 = train_model(
    model1, X_train, Y_train, T_train,
    arch="vanilla", epochs=5000, lr=5e-4,
    lam=0.1, lam_warmup=1000, norm_p=norm_p
)

RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

## FIX 5 — Evaluation (was broken: referenced undefined vars)

In [ ]:
# ── Output normalization stats for denormalization at eval ──
out_means = np.array([px_mean, py_mean, vx_mean, vy_mean, yaw_mean])
out_stds  = np.array([px_std,  py_std,  vx_std,  vy_std,  yaw_std])

# Raw (unnormalized) ground truth for scoring
Y_raw = np.stack([px_gt, py_gt, vx_gt, vy_gt, yaw_gt], axis=1)
Y_test_raw = Y_raw[n_train + n_val:]


def evaluate(model, X_te, T_te, Y_te_raw, out_means, out_stds):
    model.eval()
    device = next(model.parameters()).device
    with torch.no_grad():
        pred_norm = model(X_te.to(device), T_te.to(device))
    pred_norm = pred_norm.cpu().numpy()

    # Denormalize back to real units
    pred_real = pred_norm * out_stds + out_means
    gt = Y_te_raw

    pos_err  = np.sqrt((pred_real[:,0] - gt[:,0])**2 + (pred_real[:,1] - gt[:,1])**2)
    vel_err  = np.sqrt((pred_real[:,2] - gt[:,2])**2 + (pred_real[:,3] - gt[:,3])**2)
    head_err = np.abs(np.arctan2(np.sin(pred_real[:,4] - gt[:,4]),
                                  np.cos(pred_real[:,4] - gt[:,4])))
    return pos_err, vel_err, head_err


pe1, ve1, he1 = evaluate(model1, X_test, T_test, Y_test_raw, out_means, out_stds)

In [ ]:
def smooth(arr, window=50):
    return pd.Series(arr).rolling(window, center=True).mean().values


t_test_sec = T_test[:len(pe1)].detach().numpy().flatten()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(t_test_sec, smooth(pe1), label="Vanilla PINN", color="#2563eb")
axes[0].set_title("Position Error Over Time")
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Error (m)")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(t_test_sec, smooth(ve1), color="#2563eb", label="Vanilla")
axes[1].set_title("Velocity Error Over Time")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Error (m/s)")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(t_test_sec, smooth(np.degrees(he1)), color="#2563eb", label="Vanilla")
axes[2].set_title("Heading Error Over Time")
axes[2].set_xlabel("Time (s)")
axes[2].set_ylabel("Error (degrees)")
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("benchmark_results_v2.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
print(f"{'Model':<15} {'Pos RMSE (m)':<15} {'Vel RMSE (m/s)':<16} {'Head MAE (°)':<14}")
print("-" * 60)
for name, pe, ve, he in [("Vanilla", pe1, ve1, he1)]:
    print(f"{name:<15} {np.sqrt((pe**2).mean()):<15.4f} "
          f"{np.sqrt((ve**2).mean()):<16.4f} {np.degrees(he.mean()):<14.4f}")

In [ ]:
# ── Training loss curves ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.semilogy(hist1["epoch"], hist1["data_loss"], label="Data loss")
ax1.semilogy(hist1["epoch"], hist1["phys_loss"], label="Physics loss")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss (log)")
ax1.set_title("Training Losses"); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.semilogy(hist1["epoch"], hist1["total_loss"], label="Total loss", color="black")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss (log)")
ax2.set_title("Total Loss"); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

The physics loss will likely never hit zero and that's expected — you still have IMU noise, the 2D projection is an approximation of 3D motion, and the nearest-neighbor timestamp alignment introduces small errors. A residual of ~0.3–0.5 is reasonable.